In [2]:
import os
import sys
import torch
import tiktoken

project_root = os.path.dirname(os.path.abspath("")) # since notebook is in evaluation/
sys.path.insert(0, project_root)

import model
from model.model import GPT, GPTConfig
model.GPTConfig = GPTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"Using device: {device}")

Using device: mps


In [3]:
# Load configuration matching the notebook setup
config = GPTConfig(
    block_size=256,
    vocab_size=50257,
    n_layer=8,
    n_head=12,
    n_embd=384,
    dropout=0.1
)

# Initialize model
model = GPT(config)
model.to(device)
model.eval()

model_path = os.path.join(project_root, "training", "nanogpt_checkpoint_12.pt")
if not os.path.exists(model_path):
    print(f"Error: {model_path} not found.")
    print("Please run the notebook 'training/training_pipeline.ipynb' to train and save the model.")
else:
    # Load weights
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if "model" in checkpoint:
        model.load_state_dict(checkpoint["model"])
    else:
        model.load_state_dict(checkpoint)
    print(f"Loaded model from {model_path}")

number of parameters: 33.50M
Loaded model from /Users/idant/Developer/Projects/NanoGPT/training/nanogpt_checkpoint_12.pt


In [4]:
# Setup tokenizer
enc = tiktoken.get_encoding("gpt2")

prompt = "[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]\nYeah, look, it's 3 AM in Toronto"
print(f"\nPrompt: '{prompt}'\n")

idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
max_new_tokens = 500


Prompt: '[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]
Yeah, look, it's 3 AM in Toronto'



### Evaluation

In [5]:
print("Greedy Decoding")
out_idx = model.generate(idx, 100, top_k=1, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

Greedy Decoding
[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]
Yeah, look, it's 3 AM in Toronto (Yeah)
I'm a savage mode (Yeah), I'm the savage mode (Yeah)
I'm a savage on these niggas (Yeah)
I'ma be a savage mode (Yeah)
I'm a savage mode (Yeah)
I'm a savage mode (Yeah)
I'm a savage mode (Yeah)
I'm a savage mode (Yeah)
I'm a savage mode (Yeah)
I'm a savage mode (Yeah)
I


In [6]:
print("Temperature Sampling")
torch.manual_seed(42)
out_idx = model.generate(idx, max_new_tokens, temperature=0.7, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

Temperature Sampling
[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]
Yeah, look, it's 3 AM in Toronto
You know that we got that I came on my side
Make a dollar, niggayah said you were too much
But I was a man since they spent his momma as well
I'm not scared of the one, everybody done like him; you're only a son?
They tell me "Huh?!," and your best friend"
And you really scared to say "Hell a brother?"
No one, no, he just play on some more, yeah! (Ayy)
She lyin' 'bout he gon' be alright now but she fuckin' go with him
I don't wanna hit all these rocks 'em right back
He can make him die for a new bitch showin'
I don't care if he love up, she littin' her girl
Oh, she won't get outta here before she wantle
(One day), she cryin' but she want hope
I wanna call my place, oh
She loving her all the times
I told her you
I do

Couldn tell me she would take the money realest
Pussy is the baddestie-one, y'all
So I got the first way
We'll never see them niggas sh

In [7]:
print("Top-k Sampling")
torch.manual_seed(42)
out_idx = model.generate(idx, 100, top_k=15, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

Top-k Sampling
[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]
Yeah, look, it's 3 AM in Toronto
You see your neck on my knees
And you're not so big, and you can't come to me? (I swear I'm too high)
A-up— I'm tryna keep a mama' (Straight up), but ya (Ha-ha, hey)
But you never gonna give him out of me
If this is how much you got my ass for a bitch for me (Uh, yeah, yeah)
That'll never let that you ride


In [8]:
print("Top-p Sampling")
torch.manual_seed(42)
out_idx = model.generate(idx, 100, top_p=0.8, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

Top-p Sampling
[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]
Yeah, look, it's 3 AM in Toronto
You never need to die too long as a mill' that you know when we're gone<|endoftext|>


In [ ]:
prompt = "I was on top now the bottom"  #unconditioned prompt
print(f"\nPrompt: '{prompt}'\n")

idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
out_idx = model.generate(idx, 100, temperature=0.7, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))


Prompt: 'I was on top now the bottom'

I was on top now the bottom
Pussy, I'ma give a fuck up
Gangsta niggas around me
Bitches don't know what they all they're still talking to
These niggas can be like they scared of y'all they met?

We'll be tryna make that ten plus four-one
That's what we do and start is for here to get something (Yeah)
All motherfuckers actin', y'all are you did some help
They


### Testing for memorization (Eminem lyrics)

In [10]:
prompt = "[Genre: boom_bap] [Mood: aggressive] [Rhyme: dense_internal] [Cadence: fast]\nLook, his palms are sweaty, knees weak, arms are heavy"
idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
out_idx = model.generate(idx, 100, temperature=0.7, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

[Genre: boom_bap] [Mood: aggressive] [Rhyme: dense_internal] [Cadence: fast]
Look, his palms are sweaty, knees weak, arms are heavy
Some battle, blackements is heavy, the part of rocks and off
You're so you wanna be mad at me, can't be too
I don't be damned if I'ma with no more than livin'
And I ain't like a zombie, 'em, but it's just as no
No matter how many times this means makes my niggas got
I see the Devil in a box like a cage though he's den
So now she take a


### Contradiction Test

In [16]:
prompt = "[Genre: conscious] [Mood: peaceful] [Rhyme: simple] [Cadence: slow]\nLook, I was poor"
idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
out_idx = model.generate(idx, 500, temperature=0.7, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

[Genre: conscious] [Mood: peaceful] [Rhyme: simple] [Cadence: slow]
Look, I was poor
I started gettin' back to my mind (Yeah)
People went down
You'd never leave
But there's been lovin', mind, it might be alone
I can't take no more, and now

Don't you lie, today, just stay gone
Just in love to me now
Your better life is a love, I'm walkin' home
But I've done again
My first day that feels like I don't cry for myself
Funny so much to me
This is right there are you tonight
No leeches
Livin' on your love
Feels me now
Is this everything? What?<|endoftext|>
